In [14]:
import pandas as pd
import min_features, daily_return
import importlib
importlib.reload(min_features)
importlib.reload(daily_return)

perf_df = pd.read_csv("master_run_results.csv")
perf_df['Date'] = perf_df['test_start']
returns = [1, 2, 3, 5, 10]
df_daily = daily_return.pull_daily('QQQ', returns) 
return_cols = df_daily.columns[df_daily.columns.str.contains("Return_")].to_list()
df_returns = df_daily[['Date'] + return_cols][(df_daily['Date'] < '2025-12-20') & (df_daily['Date'] > '2025-01-01')].copy()

In [102]:
r = 5
df_returns = df_daily[['Date', 'Close'] + return_cols][(df_daily['Date'] < '2025-12-20') & (df_daily['Date'] > '2025-01-01')].copy()
df_returns_r = df_returns[['Date', 'Close', f'Return_{r}']].sort_values(by='Date').copy()
# df has columns: ["Date", "Return_1"] where Return_1 is 0/1 (or False/True)

s = df_returns_r[f"Return_{r}"].astype(int)

# identify streak groups (new group whenever value changes)
grp = s.ne(s.shift()).cumsum()

# streak length within each group: 1,2,3,...
streak_len = s.groupby(grp).cumcount() + 1

# positive streaks for 1s, negative for 0s
df_returns_r["streak"] = streak_len.where(s.eq(1), -streak_len)
df_returns_r["streak_lag1"] = df_returns_r["streak"].shift(1).fillna(0).astype("Int64")

#df_returns_1.sort_values(by='Date', ascending=False)

perf_cols = ['model', 'acc', 'Date', 'train_years', 'feature_set']
performance_2 = pd.merge(df_returns_r, perf_df[(perf_df['horizon'] == r) & (perf_df['test_days'] == 1)], on='Date', how='inner')

df = performance_2.copy()
# 1) Accuracy by (model, train_years, streak)
acc_piv = df.pivot_table(
    index=["model", "train_years", 'feature_set'],
    columns="streak_lag1",
    values="acc",
    aggfunc="mean"
)

# 2) Count by (model, train_years, streak)
cnt_piv = df.pivot_table(
    index=["model", "train_years", 'feature_set'],
    columns="streak_lag1",
    values="acc",          # any column works since we're counting rows
    aggfunc="size"
)

# 3) Combine into one wide table with clear column labels
out = pd.concat({"acc": acc_piv, "count": cnt_piv}, axis=1)

# optional: sort columns so streaks go -N ... -1, 1 ... N
out = out.reindex(sorted(out.columns, key=lambda x: (x[1] >= 0, x[1])), axis=1)

gcols = ["model", "train_years", "feature_set"]

df2 = df.copy()
df2["side"] = df2["streak"].gt(0).map({True: "pos", False: "neg"})  # 0 shouldn't exist with your streak logic

side_perf = (
    df2.groupby(gcols + ["side"])
       .agg(n=("acc", "size"), acc=("acc", "mean"))
       .reset_index()
)

# wide format (pos/neg columns)
side_wide = side_perf.pivot(index=gcols, columns="side", values=["acc", "n"])
side_wide

acc               n       
side                                         neg       pos   neg    pos
model         train_years feature_set                                  
random_forest 4           daily         0.734940  0.882759  83.0  145.0
                          daily+minute  0.554217  0.868966  83.0  145.0
                          minute        0.084337  0.889655  83.0  145.0
              6           daily         0.746988  0.889655  83.0  145.0
                          daily+minute  0.469880  0.896552  83.0  145.0
                          minute        0.072289  0.889655  83.0  145.0
xgboost       4           daily         0.650602  0.855172  83.0  145.0
                          daily+minute  0.506024  0.758621  83.0  145.0
                          minute        0.216867  0.786207  83.0  145.0
              6           daily         0.602410  0.882759  83.0  145.0
                          daily+minute  0.530120  0.834483  83.0  145.0
                          minute        0.265060  0.682759  83.0  145.0

In [104]:
K = 3
gcols = ["model", "train_years", "feature_set"]
streak_col = 'streak'#_lag1'

d = df.copy()

# keep exact -3..+3; collapse only beyond into +/-4 (representing 3+)
d["streak_bucket"] = d[streak_col].clip(lower=-K, upper=K)
d.loc[d[streak_col] < -K, "streak_bucket"] = -(K + 1)   # strictly less than -3
d.loc[d[streak_col] >  K, "streak_bucket"] =  (K + 1)   # strictly greater than +3

flip_perf = (
    d.groupby(gcols + ["streak_bucket"])
     .agg(n=("acc", "size"), acc=("acc", "mean"))
)

flip_wide = pd.concat(
    {"acc": flip_perf["acc"].unstack("streak_bucket"),
     "n":   flip_perf["n"].unstack("streak_bucket")},
    axis=1
)

# order: +1 acc, -1 acc, +1 n, -1 n, ... +3 acc, -3 acc, +3 n, -3 n, 3+ acc, -3+ acc, 3+ n, -3+ n
ordered_cols = []
for k in [1, 2, 3, "3+"]:
    pb = (K + 1) if k == "3+" else k
    nb = -(K + 1) if k == "3+" else -k
    ordered_cols += [("acc", pb), ("acc", nb), ("n", pb), ("n", nb)]

flip_wide = flip_wide.reindex(columns=pd.MultiIndex.from_tuples(ordered_cols))

# relabel buckets
rename_cols = []
for metric, b in flip_wide.columns:
    if b == (K + 1): lab = "3+"
    elif b == -(K + 1): lab = "-3+"
    else: lab = str(b)
    rename_cols.append((metric, lab))
flip_wide.columns = pd.MultiIndex.from_tuples(rename_cols)

flip_wide.round(2)

acc         n       acc         n  \
                                           1    -1   1  -1     2    -2   2   
model         train_years feature_set                                        
random_forest 4           daily         0.33  0.12  18  16  0.87  0.85  15   
                          daily+minute  0.44  0.06  18  16  0.80  0.15  15   
                          minute        0.94  0.12  18  16  0.87  0.00  15   
              6           daily         0.33  0.06  18  16  0.87  0.69  15   
                          daily+minute  0.61  0.06  18  16  0.73  0.00  15   
                          minute        1.00  0.00  18  16  0.53  0.15  15   
xgboost       4           daily         0.56  0.19  18  16  0.87  0.54  15   
                          daily+minute  0.44  0.12  18  16  0.53  0.31  15   
                          minute        0.89  0.19  18  16  0.73  0.15  15   
              6           daily         0.56  0.12  18  16  0.87  0.38  15   
                          daily+minute  0.56  0.25  18  16  0.67  0.15  15   
                          minute        0.89  0.38  18  16  0.47  0.23  15   

                                             acc         n       acc        \
                                        -2     3    -3   3  -3    3+   -3+   
model         train_years feature_set                                        
random_forest 4           daily         13  1.00  0.92  15  13  0.97  0.88   
                          daily+minute  13  1.00  0.77  15  13  0.94  0.80   
                          minute        13  0.73  0.15  15  13  0.91  0.07   
              6           daily         13  1.00  1.00  15  13  0.98  0.95   
                          daily+minute  13  0.93  0.77  15  13  0.97  0.68   
                          minute        13  0.87  0.08  15  13  0.93  0.07   
xgboost       4           daily         13  0.93  0.92  15  13  0.90  0.78   
                          daily+minute  13  0.87  0.38  15  13  0.84  0.76   
                          minute        13  0.80  0.31  15  13  0.77  0.22   
              6           daily         13  0.93  0.85  15  13  0.94  0.78   
                          daily+minute  13  0.80  0.69  15  13  0.92  0.71   
                          minute        13  0.80  0.23  15  13  0.66  0.24   

                                         n      
                                        3+ -3+  
model         train_years feature_set           
random_forest 4           daily         97  41  
                          daily+minute  97  41  
                          minute        97  41  
              6           daily         97  41  
                          daily+minute  97  41  
                          minute        97  41  
xgboost       4           daily         97  41  
                          daily+minute  97  41  
                          minute        97  41  
              6           daily         97  41  
                          daily+minute  97  41  
                          minute        97  41